In [0]:
from pyspark.sql.functions import col,when,to_date,current_timestamp
from pyspark.sql.types import StructType,StructField,StringType,IntegerType
dbutils.widgets.text("processing_date","2026-06-21","Processing Date")
proc_date = dbutils.widgets.get("processing_date")

employee_data = StructType([
    StructField("emp_id",IntegerType(),True),
    StructField("emp_name",StringType(),True),
    StructField("salary", IntegerType(), True),
    StructField("hire_date", StringType(), True)
])

In [0]:
raw_landing_data = [
    (101, "Alice Smith", 95000, "2024-01-15"),
    (102, "Bob Jones", 105000, "2023-11-20"),
    (999, "Corrupted Record", 0, "BAD-DATE-FORMAT") # Bad data row
]
df_raw = spark.createDataFrame(raw_landing_data,schema=employee_data)
df_validated = df_raw.withColumn("parsed_hire_date",to_date(col("hire_date"),"yyyy-MM-dd"))
df_clean = df_validated.filter(col("parsed_hire_date").isNotNull()).drop("hire_date")
df_corrupted = df_validated.filter(col("parsed_hire_date").isNull())
bad_record_count = df_corrupted.count()
if bad_record_count > 0:
    df_corrupted.write.format("delta").mode("overwrite").saveAsTable("quarantined_bad_records")
    print(f"⚠️ Sent {bad_record_count} corrupted records to quarantine table.")



In [0]:
from delta.tables import DeltaTable
spark.sql("""
    CREATE TABLE IF NOT EXISTS target_employees (
        emp_id INT, emp_name STRING, salary INT, parsed_hire_date DATE, updated_at TIMESTAMP
    ) USING DELTA
""")
target_delta_table  = DeltaTable.forName(spark,"target_employees")
(target_delta_table.alias("target").merge(
    df_clean.alias("source"),
    "target.emp_id =source.emp_id"
)
.whenMatchedUpdate(set = {
     "target.emp_name": "source.emp_name",
     "target.salary": "source.salary",
     "target.updated_at": "source.updated_at"
}
)
.whenNotMatchedInsertAll()
.execute()
)

In [0]:
spark.sql("OPTIMIZE target_employees ZORDER BY (emp_id)")

In [0]:
import json
response_metadata = {"status":"SUCCESS",
                     "clean_processed": df_clean.count(),
                     "bad_quarantined": bad_record_count
                     }
dbutils.notebook.exit(json.dumps(response_metadata))